# 📊 Public Procurement Risk Analysis and Risk Level Prediction
# 📊 Kamu İhale Risk Analizi ve Risk Seviyesi Tahmini

This notebook presents an end-to-end machine learning pipeline for analyzing public procurement data and predicting procurement risk levels.

Bu notebook, kamu ihale verilerinin analiz edilmesi ve ihale risk seviyelerinin makine öğrenmesi yöntemleriyle tahmin edilmesi amacıyla hazırlanmıştır.

---

## 📚 1. Libraries / Kütüphaneler

This section imports all required Python libraries used throughout the project.

Bu bölümde proje boyunca kullanılacak Python kütüphaneleri içe aktarılmaktadır.

In [ ]:
!pip install unidecode

In [ ]:
import polars as pl
import pandas as pd
import numpy as np


import matplotlib.pyplot as plt
import seaborn as sns


import nltk
import re
from unidecode import unidecode


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

## ⚙️ 2. Environment Setup / Ortam Hazırlığı

This section configures the execution environment and connects Google Drive.

Bu bölümde çalışma ortamı hazırlanmakta ve Google Drive bağlantısı kurulmaktadır.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

## 💾 3. Dataset Loading / Veri Setinin Yüklenmesi

This section loads the procurement dataset and performs initial validation.

Bu bölümde ihale veri seti yüklenmekte ve ilk kontroller gerçekleştirilmektedir.

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/Colab Notebooks/SAGE/GTI Global Public Procurement Dataset (GPPD) 12.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extract(
        "GTI Global Public Procurement Dataset (GPPD) 12/DE_DIB_2023.csv.gz",
        "/content/"
    )

    import zipfile

zip_path = "/content/drive/MyDrive/Colab Notebooks/SAGE/GTI Global Public Procurement Dataset (GPPD) 12.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extract("GTI Global Public Procurement Dataset (GPPD) 12/DE_DIB_2023.csv.gz", "/content/")

    df = pd.read_csv(
    "/content/GTI Global Public Procurement Dataset (GPPD) 12/DE_DIB_2023.csv.gz",
    compression="gzip"
)

    df.shape

## 🔍 4. Exploratory Data Analysis (EDA) / Keşifsel Veri Analizi

This section explores the dataset using descriptive statistics and visual analysis.

Bu bölümde veri setinin genel yapısı istatistiksel ve görsel yöntemlerle incelenmektedir.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
import matplotlib.pyplot as plt

missing = (
    df.isnull()
    .mean()
    .sort_values(ascending=False) * 100
)

missing = missing[missing > 0]

plt.figure(figsize=(16,6))

plt.bar(
    missing.index,
    missing.values
)

plt.xticks(rotation=90)
plt.ylabel("Eksik Veri (%)")
plt.xlabel("Kolonlar")
plt.title("Kolon Bazında Eksik Veri Oranları")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

dtype_counts = df.dtypes.value_counts()

plt.figure(figsize=(8,5))

plt.bar(
    dtype_counts.index.astype(str),
    dtype_counts.values
)

plt.xlabel("Veri Tipleri")
plt.ylabel("Kolon Sayısı")
plt.title("Veri Tiplerinin Dağılımı")

plt.show()

## 🧹 5. Data Preprocessing / Veri Ön İşleme

This section cleans the dataset and prepares it for feature engineering.

Bu bölümde veri temizlenmekte ve özellik mühendisliği için hazır hale getirilmektedir.


In [ ]:
# Identifying columns with a single value
# Tek değer içeren kolonları belirleme

constant_analysis = df.nunique(dropna=False).sort_values()

constant_analysis
constant_analysis[constant_analysis == 1]

In [ ]:
constant_cols = [
    "lot_validbidscount",
    "cancellation_reason",
    "filter_losingbids",
    "filter_opentender",
    "filter_framework",
    "filter_cancelled"
]
df = df.drop(columns=constant_cols)

In [ ]:
df.shape

In [ ]:
# Selecting columns that are more than 90% missing
import matplotlib.pyplot as plt


missing_table = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df)) * 100
})

# Selecting columns with more than 90% missing values
missing_over_90 = missing_table[
    missing_table["missing_percent"] > 90
].sort_values(
    by="missing_percent",
    ascending=False
)


display(missing_over_90)


plt.figure(figsize=(12, 6))

plt.bar(
    missing_over_90.index,
    missing_over_90["missing_percent"]
)

plt.xlabel("Kolonlar")
plt.ylabel("Eksik Veri Yüzdesi (%)")
plt.title("%90'dan Fazla Eksik Veri İçeren Kolonlar")

plt.xticks(
    rotation=90,
    ha="right"
)

for i, value in enumerate(missing_over_90["missing_percent"]):
    plt.text(
        i,
        value + 1,
        f"{value:.1f}%",
        ha="center",
        fontsize=9
    )

plt.ylim(0, 100)
plt.tight_layout()
plt.show()



In [ ]:
# Deleting columns with over 90% missing values
# Eksik veri oranı %90'ın üzerinde olan kolonları veri setinden çıkarma

cols_to_drop_missing = [
    'lot_updateddurationdays',
    'tender_contractsignaturedate',
    'bid_subcontractedproportion',
    'bidder_contactName',
    'corr_tax_haven',
    'tender_cancellationdate',
    'bidder_sourceid_type',
    'bidder_extra_source_id',
    'bidder_id'
]

df = df.drop(columns=cols_to_drop_missing)

df.shape




In [ ]:
# Missing data analysis for the remaining columns

missing_remaining = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df)) * 100
})

missing_remaining = missing_remaining[
    missing_remaining["missing_count"] > 0
].sort_values(
    by="missing_percent",
    ascending=False
)

display(missing_remaining)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,6))

plt.bar(
    missing_remaining.index,
    missing_remaining["missing_percent"]
)

plt.xlabel("Kolonlar")
plt.ylabel("Eksik Veri Yüzdesi (%)")
plt.title("Temizleme Sonrası Kalan Eksik Veri Oranları")

plt.xticks(
    rotation=90,
    ha="right"
)

for i, value in enumerate(missing_remaining["missing_percent"]):
    plt.text(
        i,
        value + 0.5,
        f"{value:.1f}%",
        ha="center",
        fontsize=8
    )

plt.ylim(0,100)
plt.tight_layout()

plt.show()

In [ ]:
# Yüksek eksik oranına sahip ve modelleme için anlamlı olmayan kolonların silinmesi
# Dropping columns with high missing value rates that are not meaningful for modeling

cols_to_drop_missing = [
    'lot_estimatedpriceUsd',       # Lot tahmini fiyatı (USD), %89.66 eksik
    'lot_estimatedprice',          # Lot tahmini fiyatı, %89.65 eksik
    'lot_smebidscount',            # Lot üzerindeki KOBİ teklif sayısı, %87.64 eksik
    'bidder_url',                  # Teklif veren kurum web adresi, %87.47 eksik
    'lot_electronicbidscount',     # Elektronik teklif sayısı, %82.42 eksik
    'bidder_email',                # Teklif veren e-posta bilgisi, %76.63 eksik
    'bidder_phone',                # Teklif veren telefon bilgisi, %75.71 eksik
    'buyer_sourceid_type',         # Alıcı kaynak kimlik tipi, %74.96 eksik
    'buyer_extra_source_id',       # Alıcı ek kaynak kimliği, %74.87 eksik
    'bid_isconsortium',            # Konsorsiyum teklif bilgisi, %73.57 eksik
    'buyer_id',                    # Alıcı kurum kimliği, %67.87 eksik
    'bid_digiwhist_price',         # Digiwhist teklif fiyatı, %67.67 eksik
    'lot_row_nr',                  # Lot sıra numarası, %63.54 eksik
    'bidder_street'                # Teklif veren adres bilgisi, %53.29 eksik
]

df = df.drop(
    columns=cols_to_drop_missing,
    errors="ignore"
)

df.shape

In [ ]:
# Analysis of missing data in columns remaining after cleaning
# Temizleme sonrası kalan kolonlarda eksik veri analizi

missing_remaining = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df)) * 100
})

missing_remaining = missing_remaining[
    missing_remaining["missing_count"] > 0
].sort_values(
    by="missing_percent",
    ascending=False
)

missing_remaining

In [ ]:
cols_to_keep = [
    'tender_estimatedprice',          # %83.72 eksik — İhalenin tahmini maliyetini gösterir, ihale büyüklüğü ve fiyat analizi için önemli bir özelliktir.

    'tender_isawarded',               # %71.19 eksik — İhalenin sonuçlanıp sonuçlanmadığını gösterir, ihale sürecinin durumu hakkında bilgi sağlar.

    'bid_price',                      # %70.82 eksik — Teklif verenlerin sunduğu fiyat bilgisidir, fiyat analizi ve rekabet değerlendirmesi için önemlidir.

    'tender_finalprice',              # %62.49 eksik — İhalenin gerçekleşen sonuç fiyatıdır, tahmini fiyat ile karşılaştırma yapılmasını sağlar.

    'corr_buyer_concentration',       # %61.63 eksik — Alıcı kurum yoğunlaşması risk göstergesidir, rekabet ve şeffaflık analizi için kullanılır.

    'corr_singleb',                   # %54.20 eksik — Tek teklif risk göstergesidir, düşük rekabet durumlarını belirlemek için önemlidir.

    'lot_bidscount',                  # %54.12 eksik — İhale kalemine gelen teklif sayısını gösterir, rekabet seviyesini ölçmek için kullanılır.

    'decision_period',                # %53.22 eksik — İhale karar süresini gösterir, süreç gecikmelerini ve olağan dışı durumları analiz etmek için kullanılır.

    'tender_digiwhist_price',         # %53.03 eksik — Digiwhist tarafından sağlanan fiyat bilgisidir, fiyat karşılaştırması ve anomali analizi için değerlendirilebilir.

    'tender_awarddecisiondate',       # %50.31 eksik — İhale karar tarihidir, tarih tabanlı özellikler (yıl, ay, süre vb.) üretmek için kullanılır.

    'bid_issubcontracted',            # %49.03 eksik — Teklifte alt yüklenici kullanılıp kullanılmadığını gösterir, tedarik zinciri risk analizi için değerlidir.

    'tender_isjointprocurement',      # %48.96 eksik — İhalenin ortak satın alma kapsamında olup olmadığını gösterir, ihale yapısı hakkında bilgi sağlar.

    'tender_addressofimplementation_nuts' # %48.83 eksik — İhalenin uygulama bölgesini gösterir, bölgesel analiz ve risk farklılıkları için kullanılabilir.
]

In [ ]:
cols_to_drop_missing_2 = [
    'tender_estimatedpriceUsd',       # İhalenin tahmini fiyatının USD cinsinden karşılığı
    'bid_priceUsd',                   # Teklif fiyatının USD cinsinden karşılığı
    'tender_finalpriceUsd',           # İhale sonucundaki kesin fiyatın USD cinsinden karşılığı
    'bidder_postcode_e',              # Teklif veren kurumun elektronik posta kodu bilgisi
    'bidder_postcode',                # Teklif veren kurumun posta kodu bilgisi
    'lot_title',                      # İhale kaleminin başlığı/açıklaması
    'bidder_nuts_e_clean',            # Teklif veren kurumun temizlenmiş NUTS bölge bilgisi
    'bidder_nuts',                    # Teklif veren kurumun NUTS bölge kodu
    'bidder_nuts_3',                  # Teklif veren kurumun detaylı NUTS seviye 3 bölge bilgisi
    'bidder_nuts_2',                  # Teklif veren kurumun NUTS seviye 2 bölge bilgisi
    'bidder_nuts_1',                  # Teklif veren kurumun NUTS seviye 1 bölge bilgisi
    'bidder_city'                     # Teklif veren kurumun bulunduğu şehir bilgisi
]
df = df.drop(
    columns=cols_to_drop_missing_2,
    errors="ignore"
)

df.shape


In [ ]:
# Analysis of missing data in columns remaining after cleaning
# Temizleme sonrası kalan kolonlarda eksik veri analizi

missing_analysis = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": (df.isnull().sum() / len(df)) * 100
})


missing_analysis = missing_analysis[
    missing_analysis["missing_count"] > 0
].sort_values(
    by="missing_percent",
    ascending=False
)

missing_analysis

In [ ]:
cols_to_drop = [
    'bidder_masterid',        # Teklif veren kuruma ait benzersiz kimlik bilgisidir, modelin ezber yapmasına neden olabilir.

    'bid_id',                 # Teklif kaydının benzersiz ID bilgisidir, tahmin için anlamlı özellik içermez.

    'bidder_name',            # Teklif veren kurum adıdır, yüksek cardinality oluşturur ve genelleme kabiliyetini azaltabilir.

    'tender_publications_lastcontractawardurl'
                              # Son sözleşme ilan URL bilgisidir, doğrudan analitik değer taşımaz.
]

df = df.drop(
    columns=cols_to_drop,
    errors="ignore"
)

df.shape

In [ ]:
# Listing all columns in the dataset
# Veri setindeki tüm kolonları listeleme

df.columns.tolist()

In [ ]:
# Identifying columns containing date information in the dataset
# Veri setindeki tarih bilgisi içeren kolonları tespit etme


date_candidates = []

for col in df.columns:
    if "date" in col.lower() or "deadline" in col.lower():
        date_candidates.append(col)

date_candidates

In [ ]:
date_cols = [
    'tender_biddeadline',
    'tender_awarddecisiondate',
    'tender_publications_firstdcontractawarddate',
    'tender_publications_firstcallfortenderdate'
]

df[date_cols].dtypes

In [ ]:
# Convert the date columns to datetime format.
# Tarih sütunlarını datetime formatına dönüştürme

date_cols = [
    'tender_biddeadline',
    'tender_awarddecisiondate',
    'tender_publications_firstdcontractawarddate',
    'tender_publications_firstcallfortenderdate'
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )


In [ ]:
df[date_cols].dtypes

In [ ]:
# Creating year and month information from date columns as new features
# Tarih kolonlarından yıl ve ay bilgilerini yeni özellikler olarak oluşturma

for col in date_cols:
    df[col + "_year"] = df[col].dt.year    # Tarih bilgisinden yıl bilgisini çıkarma
    df[col + "_month"] = df[col].dt.month  # Tarih bilgisinden ay bilgisini çıkarma

| Kolon                      | Açıklama                                                                                                    |
| -------------------------- | ----------------------------------------------------------------------------------------------------------- |
| `corr_singleb`             | **Single bidder (tek teklif) riski** — İhaleye tek teklif gelmesi durumunu gösterir.                        |
| `corr_proc`                | **Prosedür riski** — İhale prosedürünün riskli olup olmadığını gösteren göstergedir.                        |
| `corr_subm`                | **Submission (teklif sunumu) riski** — Teklif süreciyle ilgili risk göstergesidir.                          |
| `corr_nocft`               | **No Call For Tender riski** — Açık çağrı yapılmadan gerçekleşen süreçleri gösteren risk göstergesidir.     |
| `corr_decp`                | **Decision period riski** — Karar süresiyle ilgili risk göstergesidir.                                      |
| `corr_buyer_concentration` | **Alıcı yoğunlaşması riski** — Belirli alıcıların belirli tedarikçilerle yoğun çalışması durumunu gösterir. |


## ⚙️ 6. Missing Value Handling / Eksik Değer Doldurma

This section handles missing values in the dataset using appropriate imputation techniques to improve data quality and prepare the dataset for further analysis and modeling.

Bu bölümde veri setindeki eksik değerler uygun doldurma yöntemleri kullanılarak ele alınmakta, veri kalitesi artırılmakta ve sonraki analiz/modelleme aşamalarına hazırlık yapılmaktadır.

In [ ]:
# Kategorik (object) veri tipindeki kolonları belirleme ve eksik değerleri "Unknown" etiketi ile doldurma
# Identifying columns with the 'object' data type and filling missing values ​​with the label "Unknown"

object_cols = df.select_dtypes(include="object").columns

df[object_cols] = df[object_cols].fillna("Unknown")

In [ ]:
# Checking for missing values ​​in categorical columns
# Kategorik kolonlarda eksik değer kalıp kalmadığını kontrol etme

df[object_cols].isnull().sum().sum()

In [ ]:
# Identifying columns with numeric (integer and float) data types and analyzing the number of missing values
# Sayısal (integer ve float) veri tipindeki kolonları belirleme ve eksik değer sayılarını analiz etme

numeric_cols = df.select_dtypes(
    include=["int64", "float64"]
).columns


numeric_missing = df[numeric_cols].isnull().sum()


numeric_missing = numeric_missing[
    numeric_missing > 0
].sort_values(ascending=False)
numeric_missing

In [ ]:
# Checking for missing values ​​in columns containing risk indicators
# Risk göstergelerini içeren kolonlarda eksik değer kontrolü yapma


corr_cols = [
    "corr_buyer_concentration",  # Alıcı yoğunlaşması risk göstergesi
    "corr_singleb",              # Tek teklifli ihale risk göstergesi
    "corr_subm"                  # Teklif sunma süreci ile ilgili risk göstergesi
]


df[corr_cols].isnull().sum()

In [ ]:
# Filling missing values ​​in risk indicator columns with -1 ( Risk göstergesi kolonlarındaki eksik değerleri -1 ile doldurma)
# Missing values ​​in these columns do not necessarily imply "no risk"; rather, they indicate a lack of data (Bu kolonlarda eksik değerler "risk yok" anlamına gelmeyebilir, veri bilgisinin bulunmadığını gösterir.)
# Therefore, assigning a value of -1 instead of 0 enables the model to distinguish missing records from actual risk values. (Bu nedenle 0 kullanmak yerine -1 değeri atanarak modelin eksik kayıtları gerçek risk değerlerinden ayırt etmesi sağlanır.)

corr_cols = [
    "corr_buyer_concentration",  # Alıcı yoğunlaşması risk göstergesi
    "corr_singleb",              # Tek teklifli ihale risk göstergesi
    "corr_subm"                  # Teklif sunma süreci risk göstergesi
]

df[corr_cols] = df[corr_cols].fillna(-1)

In [ ]:
df[corr_cols].isnull().sum()

In [ ]:
# Filling missing values ​​in the 'number of offers' column with 0
# Teklif sayısı kolonundaki eksik değerleri 0 ile doldurma


count_cols = [
    "lot_bidscount"   # İhale kalemine gelen toplam teklif sayısı
]

df[count_cols] = df[count_cols].fillna(0)

In [ ]:
df[count_cols].isnull().sum()

In [ ]:
# Converting the 'number of offers' column to the integer data type
# Teklif sayısı kolonunu integer (tam sayı) veri tipine dönüştürme

df[count_cols] = df[count_cols].astype(int)

In [ ]:
# Checking for missing values ​​in columns containing price information
# Fiyat bilgisi içeren kolonlardaki eksik değer sayılarını kontrol etme


price_cols = [

    "tender_estimatedprice",        # İhalenin tahmini fiyatı

    "bid_price",                    # Teklif verenin sunduğu fiyat

    "tender_finalprice",            # İhalenin gerçekleşen sonuç fiyatı

    "tender_digiwhist_price"        # Digiwhist fiyat göstergesi
]

df[price_cols].isnull().sum()

In [ ]:
# Filling missing values ​​in the price columns with the median value
# Fiyat kolonlarındaki eksik değerleri medyan değeri ile doldurma

for col in price_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df[price_cols].isnull().sum()

In [ ]:
# Checking for missing values ​​in columns containing duration information
# Süre bilgisi içeren kolonlardaki eksik değerleri kontrol etme

duration_cols = [
    "decision_period",      # İhale karar süresi (gün)
    "submission_period"     # Teklif verme süresi (gün)
]

df[duration_cols].isnull().sum()

In [ ]:
# Filling missing values ​​in the duration columns with the median value
# Süre kolonlarındaki eksik değerleri medyan değeri ile doldurma

for col in duration_cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df[duration_cols].isnull().sum()

In [ ]:
# Imputing missing values ​​in year and month features derived from date columns
# Tarih kolonlarından oluşturulan yıl ve ay özelliklerindeki eksik değerleri doldurma

date_feature_cols = [
    "tender_awarddecisiondate_year",
    "tender_awarddecisiondate_month",
    "tender_publications_firstdcontractawarddate_year",
    "tender_publications_firstdcontractawarddate_month",
    "tender_biddeadline_year",
    "tender_biddeadline_month",
    "tender_publications_firstcallfortenderdate_year",
    "tender_publications_firstcallfortenderdate_month"
]

df[date_feature_cols] = df[date_feature_cols].fillna(-1)

In [ ]:
df[date_feature_cols].isnull().sum()

In [ ]:
# Checking the dataset for missing values
# Veri setinde eksik değer kalıp kalmadığını kontrol etme

missing_check = df.isnull().sum().sort_values(ascending=False)

missing_check[missing_check > 0]

## 📦 7. Outlier Analysis / Aykırı Değer Analizi

This section identifies outliers and applies suitable transformations.

Bu bölümde aykırı değerler incelenmekte ve gerekli dönüşümler uygulanmaktadır.

In [ ]:
# Aykırı değer (outlier) analizi yapılacak sayısal kolonları belirleme
# Identifying numerical columns for outlier analysis


outlier_cols = [
    "tender_estimatedprice",                  # İhalenin tahmini fiyatı
    "tender_finalprice",                      # İhalenin sonuçlanan gerçek fiyatı
    "bid_price",                              # Teklif veren firmanın sunduğu fiyat
    "tender_digiwhist_price",                 # Digiwhist tarafından hesaplanan fiyat göstergesi

    "tender_recordedbidscount",               # İhaleye kayıtlı toplam teklif sayısı
    "lot_bidscount",                          # İhale kalemine gelen teklif sayısı

    "submission_period",                      # Teklif verme süresi (gün)
    "decision_period",                        # İhale karar süresi (gün)
]

In [ ]:
outlier_results = []

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_count = (
        (df[col] < lower) |
        (df[col] > upper)
    ).sum()

    outlier_percent = (outlier_count / len(df)) * 100

    outlier_results.append({
        "Kolon": col,
        "Alt Sınır": lower,
        "Üst Sınır": upper,
        "Uç Değer Sayısı": outlier_count,
        "Uç Değer Oranı (%)": round(outlier_percent,2)
    })

outlier_df = pd.DataFrame(outlier_results)

outlier_df

In [ ]:
# Visualizing outlier rates
# Outlier oranlarını görselleştirme

import matplotlib.pyplot as plt


plt.figure(figsize=(10,6))

plt.barh(
    outlier_df["Kolon"],
    outlier_df["Uç Değer Oranı (%)"]
)

plt.xlabel("Uç Değer Oranı (%)")
plt.ylabel("Kolonlar")
plt.title("Sayısal Değişkenlerde Aykırı Değer Oranları")

for index, value in enumerate(outlier_df["Uç Değer Oranı (%)"]):
    plt.text(
        value + 0.5,
        index,
        f"%{value}",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
# Sayısal değişkenlerde çarpıklık (skewness) kontrolü
# skewness < 1 → normal kabul edilebilir
# 1 - 2 → orta derecede sağ çarpık
# >2 → güçlü sağ çarpık, log dönüşümü önerilir

skewness = df[outlier_cols].skew().sort_values(ascending=False)

skewness

In [ ]:
# Sağa çarpık dağılıma sahip sayısal değişkenlere log dönüşümü uygulama
# Log dönüşümü uç değerleri veri setinden kaldırmaz; büyük değerlerin etkisini azaltarak
# dağılımı daha dengeli hale getirir. np.log1p() fonksiyonu kullanılarak
# 0 değeri içeren kayıtların da güvenli bir şekilde dönüştürülmesi sağlanmaktadır.

import numpy as np

df["log_tender_recordedbidscount"] = np.log1p(
    df["tender_recordedbidscount"]
)

df["log_lot_bidscount"] = np.log1p(
    df["lot_bidscount"]
)

df["log_decision_period"] = np.log1p(
    df["decision_period"]
)

df["log_tender_estimatedprice"] = np.log1p(
    df["tender_estimatedprice"]
)

df["log_tender_finalprice"] = np.log1p(
    df["tender_finalprice"]
)

df["log_submission_period"] = np.log1p(
    df["submission_period"]
)

## ⚙️ 8. Feature Engineering / Özellik Mühendisliği

This section creates new informative features to improve model performance.

Bu bölümde model performansını artırmak amacıyla yeni özellikler oluşturulmaktadır.

In [ ]:
# "Bir ihale ilan edildikten sonra karar verilmesi ne kadar sürdü?"

df["decision_duration_days"] = (
    df["tender_awarddecisiondate"]
    - df["tender_publications_firstcallfortenderdate"]
).dt.days


In [ ]:
df["decision_duration_days"].isnull().sum()

In [ ]:
# Karar süresi değişkeninde negatif değer bulunup bulunmadığını kontrol etme

(df["decision_duration_days"] < 0).sum()

In [ ]:
# Karar süresi negatif olan kayıtları eksik değer (NaN) olarak işaretleme
# Karar süresi negatif olamayacağı için bu kayıtlar veri tutarsızlığı olarak değerlendirilmiştir.

df.loc[
    df["decision_duration_days"] < 0,
    "decision_duration_days"
] = np.nan

In [ ]:
# Değişkenin dağılımı analiz edilerek veri seti için uygun maksimum eşik değeri belirlenmekte
# ve olası uç değerlerin değerlendirilmesi amaçlanmaktadır.

df["decision_duration_days"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]
)

#decision_duration_days değişkeninin ortalaması 143.82 gün, medyanı ise 102 gündür.
#Ortalama değerin medyandan yüksek olması dağılımın sağa çarpık olduğunu göstermektedir.
#Değişkenin %99.9'u yaklaşık 1356 gün içerisinde tamamlanırken, maksimum değer 5284 gündür.
#Bu uzun süreler istatistiksel olarak sıra dışı olsa da kamu ihalerinin doğası gereği gerçek süreçleri temsil edebileceğinden veri setinden çıkarılmamıştır.

In [ ]:
risk_cols = [
    "corr_nocft",
    "corr_proc",
    "corr_decp",
    "corr_singleb",
    "corr_subm",
    "corr_buyer_concentration",
]

df[risk_cols].describe()

In [ ]:
# İkili risk göstergelerinde eksik değerleri temsil eden -1 değerlerini 0 ile değiştirme
# Bu kolonlar risk var/yok bilgisini temsil eden binary değişkenlerdir.
# -1 değeri eksik veya bilinmeyen durumları ifade ettiği için modelleme aşamasında
# hatalı bir risk değeri olarak algılanmaması amacıyla 0 (risk yok) olarak değiştirilmiştir.

binary_risk_cols = [
    "corr_nocft",      # İhale çağrısı yapılmadan gerçekleştirilen ihale riski göstergesi
    "corr_proc",       # Uygulanan ihale prosedürüne bağlı risk göstergesi
    "corr_decp",       # İhale karar sürecindeki gecikme veya düzensizlik riski göstergesi
    "corr_singleb",    # Tek teklif alınması nedeniyle oluşan rekabet eksikliği riski göstergesi
    "corr_subm"        # Teklif verme süresi ile ilişkili risk göstergesi
]

df[binary_risk_cols] = df[binary_risk_cols].replace(-1,0)

In [ ]:
# Eksik bilgi durumunu ayrı bir özellik olarak oluşturma
# corr_subm ve corr_buyer_concentration kolonlarında -1 değeri gerçek bir risk değeri değil,
# ilgili bilginin eksik veya hesaplanamamış olduğunu göstermektedir.
# Bu nedenle eksiklik bilgisi kaybolmaması için ayrı binary kolonlar oluşturulmuştur.
# Model böylece hem risk bilgisini hem de ilgili bilginin eksik olma durumunu öğrenebilir.

df["subm_missing"] = (
    df["corr_subm"] == -1
).astype(int)

# subm_missing: Teklif verme süresi (submission period) risk bilgisinin eksik olup olmadığını gösterir.
# 1 → corr_subm bilgisi mevcut değil
# 0 → corr_subm bilgisi mevcut


df["buyer_concentration_missing"] = (
    df["corr_buyer_concentration"] == -1
).astype(int)
# buyer_concentration_missing: Alıcı yoğunlaşması risk bilgisinin eksik olup olmadığını gösterir.
# 1 → alıcı yoğunlaşması bilgisi hesaplanamamış
# 0 → alıcı yoğunlaşması bilgisi mevcut

In [ ]:
# Risk göstergelerinde eksik olarak temsil edilen -1 değerlerini 0 ile değiştirme
# Bu işlem ile asıl risk kolonları modelleme için uygun binary/sayısal forma dönüştürülmektedir.
# Böylece model -1 değerini ayrı bir risk seviyesi olarak algılamaz.

# Teklif süresi risk göstergesindeki eksik değerleri risk yok olarak güncelleme
# -1 → bilgi eksik, 0 → risk yok, 1 → risk var anlamındadır.
df["corr_subm"] = df["corr_subm"].replace(-1,0)


# Alıcı yoğunlaşması risk göstergesindeki eksik değerleri 0 ile değiştirme
# Eksik durum bilgisi "buyer_concentration_missing" kolonunda tutulduğu için
# ana kolon modelleme aşamasında kullanılabilir hale getirilmiştir.
df["corr_buyer_concentration"] = (
    df["corr_buyer_concentration"]
    .replace(-1,0)
)

In [ ]:
# İhale bazında toplam risk göstergesi sayısını oluşturma


risk_cols = [
    "corr_nocft",                # İhale çağrısı yapılmaması riski
    "corr_proc",                 # İhale prosedürü kaynaklı risk
    "corr_decp",                 # Karar süreci riski
    "corr_singleb",              # Tek teklif nedeniyle rekabet riski
    "corr_subm",                 # Teklif süresi kaynaklı risk
    "corr_buyer_concentration"   # Alıcı yoğunlaşması riski
]

df["risk_indicator_count"] = df[risk_cols].sum(axis=1)

In [ ]:
# En yüksek toplam risk göstergesine sahip ilk 10 ihaleyi görüntüleme


df[[
    "risk_indicator_count",       # İhalede bulunan toplam risk göstergesi sayısı
    "corr_nocft",                 # İhale çağrısı yapılmaması riski
    "corr_proc",                  # Prosedür kaynaklı risk
    "corr_decp",                  # Karar süreci riski
    "corr_singleb",               # Tek teklif nedeniyle rekabet riski
    "corr_subm",                  # Teklif süresi kaynaklı risk
    "corr_buyer_concentration"    # Alıcı yoğunlaşması riski
]].sort_values(
    by="risk_indicator_count",
    ascending=False
).head(10)

In [ ]:
# Rekabet seviyesini ve teklif varlığını gösteren yeni özelliklerin oluşturulması
# İhale sürecinde teklif sayısı, rekabet düzeyinin önemli göstergelerinden biridir.
# Bu nedenle teklif sayısından yararlanılarak iki yeni binary özellik oluşturulmuştur.

# Düşük rekabet göstergesi oluşturma
# Bir ihaleye 1 veya daha az teklif gelmesi düşük rekabet durumu olarak kabul edilmiştir.
# 1 → düşük rekabet var
# 0 → yeterli rekabet var
df["low_competition"] = (
    df["tender_recordedbidscount"] <= 1
).astype(int)


# İhalede teklif bulunup bulunmadığını gösteren özellik oluşturma
# Teklif sayısının sıfırdan büyük olması, ihaleye en az bir teklif geldiğini gösterir.
# 1 → teklif mevcut
# 0 → teklif bulunmuyor
df["has_bids"] = (
    df["tender_recordedbidscount"] > 0
).astype(int)

In [ ]:
# En uzun %10 karar süreçlerini risk göstergesi olarak işaretle
df["long_decision_duration"] = (
    df["decision_duration_days"] >=
    df["decision_duration_days"].quantile(0.90)
).astype(int)


# En kısa %10 teklif sürelerini risk göstergesi olarak işaretle
df["short_submission_period"] = (
    df["submission_period"] <=
    df["submission_period"].quantile(0.10)
).astype(int)


# Toplam ihale süreç süresi
df["total_process_duration"] = (
    df["submission_period"] +
    df["decision_period"]
)

In [ ]:
import numpy as np

# Karar süresi değişkeninin log dönüşümü
# Uzun kuyruklu dağılımın etkisini azaltmak için uygulanır.
df["log_decision_duration_days"] = np.log1p(
    df["decision_duration_days"]
)


# Toplam ihale süreç süresinin log dönüşümü
# Aşırı uzun süreçlerin model üzerindeki etkisini azaltmak için uygulanır.
df["log_total_process_duration"] = np.log1p(
    df["total_process_duration"]
)

In [ ]:
# Eksik değer kontrolü
df["log_decision_duration_days"].isnull().sum()

In [ ]:
# Log dönüşümü uygulanmış karar süresi değişkenindeki eksik değerleri
# medyan değer ile doldurma
# Medyan kullanımı, uzun süreçlerin oluşturduğu uç değer etkisini azaltır.

df["log_decision_duration_days"] = (
    df["log_decision_duration_days"]
    .fillna(df["log_decision_duration_days"].median())
)

In [ ]:
# Risk göstergesi oluştururken kullanılan eşik değerlerini kontrol etme
# Teklif süresi için %10 persentil değeri, en kısa teklif sürelerini belirlemek;
# karar süresi için %90 persentil değeri ise en uzun karar süreçlerini belirlemek amacıyla kullanılmıştır.

print("submission %10:", df["submission_period"].quantile(0.10))
print("decision %90:", df["decision_duration_days"].quantile(0.90))

In [ ]:
print(df["short_submission_period"].value_counts())

print(df["long_decision_duration"].value_counts())

## 🎯 7. Feature Selection / Özellik Seçimi

This section selects the most informative features for machine learning models by removing redundant, irrelevant, and highly correlated variables.

Bu bölümde makine öğrenmesi modellerinde kullanılacak en anlamlı özellikler belirlenmektedir. Gereksiz, tekrar eden ve model performansını olumsuz etkileyebilecek değişkenler analiz edilerek uygun özellikler seçilmektedir.

In [ ]:
model_features = [

    # Temel ihale bilgileri
    "tender_proceduretype",
    "tender_supplytype",
    "buyer_country",
    "buyer_mainactivities",

    # Rekabet bilgileri
    "log_tender_recordedbidscount",
    "log_lot_bidscount",
    "low_competition",
    "has_bids",

    # Risk bilgileri
    "risk_indicator_count",
    "corr_nocft",
    "corr_proc",
    "corr_decp",
    "corr_singleb",
    "corr_subm",
    "corr_buyer_concentration",

    # Süre bilgileri
    "log_submission_period",
    "log_decision_period",
    "log_decision_duration_days",
    "short_submission_period",
    "long_decision_duration",
    "log_total_process_duration",

    # Fiyat bilgileri
    "log_tender_estimatedprice",
    "log_tender_finalprice"
]

In [ ]:
# Seçilen özelliklerden yeni bir veri seti oluşturma
# Bu işlem ile modelleme aşamasında kullanılacak olan önemli değişkenler
# ana veri setinden ayrılarak yeni bir DataFrame oluşturulur.
# copy() kullanılarak orijinal df veri setinden bağımsız bir kopya alınır,
# böylece yapılacak değişikliklerin ana veri setini etkilemesi önlenir.

df_model = df[model_features].copy()

In [ ]:
df_model.shape

In [ ]:
# Veri setindeki sayısal değişkenleri seçme
# Korelasyon analizi yalnızca sayısal değişkenler üzerinde uygulanabildiği için
# object ve kategorik değişkenler analiz dışında bırakılmıştır.

numeric_df = df_model.select_dtypes(
    include=["int64", "float64"]
)


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np




corr_matrix = numeric_df.corr(method="pearson")



mask = np.triu(
    np.ones_like(corr_matrix, dtype=bool)
)


plt.figure(figsize=(18,14))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Pearson Correlation"}
)


plt.title(
    "Özellikler Arası Pearson Korelasyon Matrisi",
    fontsize=16
)


plt.xticks(
    rotation=90,
    fontsize=9
)

plt.yticks(
    rotation=0,
    fontsize=9
)


plt.tight_layout()

plt.show()

In [ ]:
import plotly.express as px

fig = px.histogram(
    df_model,
    x="risk_indicator_count",
    nbins=10,
    title="Risk Gösterge Sayısının Dağılımı",
    labels={
        "risk_indicator_count": "Risk Gösterge Sayısı"
    }
)

fig.show()

## 🤖 9. Data Preparation for Machine Learning / Makine Öğrenmesi İçin Veri Hazırlama

This section prepares the selected features for machine learning algorithms by applying preprocessing steps such as encoding categorical variables, scaling numerical features, and splitting the dataset into training and test sets.

Bu bölümde seçilen özellikler, makine öğrenmesi algoritmalarında kullanılmak üzere hazırlanmaktadır. Kategorik değişkenler kodlanmakta, sayısal değişkenler ölçeklendirilmekte ve veri seti eğitim ile test kümelerine ayrılarak modelleme aşamasına hazır hale getirilmektedir.

In [ ]:
# Risk seviyelerini belirleyen fonksiyon oluşturma
# risk_indicator_count değişkenindeki toplam risk göstergesi sayısına göre
# her ihale için kategorik bir risk seviyesi oluşturulmaktadır.
# Bu sınıflandırma, sürekli bir risk skorunu makine öğrenmesi modellerinde
# hedef değişken (target variable) olarak kullanılabilecek kategorik bir yapıya dönüştürür.

def risk_category(x):
    # 0 veya 1 adet risk göstergesine sahip ihaleler düşük risk olarak sınıflandırılır.
    if x <= 1:
        return "Düşük Risk"

    # 2 adet risk göstergesine sahip ihaleler orta risk olarak sınıflandırılır.
    elif x <= 2:
        return "Orta Risk"

    # 3 ve üzeri risk göstergesine sahip ihaleler yüksek risk olarak sınıflandırılır.
    else:
        return "Yüksek Risk"


# Her ihale için risk seviyesi oluşturma
# risk_indicator_count değerleri fonksiyona gönderilerek her kayıt için
# Düşük, Orta veya Yüksek Risk etiketi atanır.
# Oluşturulan risk_level değişkeni modelleme aşamasında tahmin edilecek hedef değişken olarak kullanılacaktır.

df_model["risk_level"] = df_model["risk_indicator_count"].apply(risk_category)

In [ ]:
df_model["risk_level"].value_counts()

In [ ]:
import plotly.express as px

risk_distribution = (
    df_model["risk_level"]
    .value_counts()
    .reset_index()
)

risk_distribution.columns = ["Risk Seviyesi", "İhale Sayısı"]

fig = px.pie(
    risk_distribution,
    names="Risk Seviyesi",
    values="İhale Sayısı",
    title="İhale Risk Seviyesi Dağılımı",
    hole=0.4
)

fig.show()

In [ ]:
import plotly.express as px

fig = px.box(
    df_model,
    x="risk_level",
    y="log_lot_bidscount",
    title="Risk Seviyelerine Göre Teklif Sayısı Dağılımı"
)

fig.show()

In [ ]:
# Bağımsız değişkenler (X) ve hedef değişkenin (y) ayrılması
# Makine öğrenmesi modellerinde kullanılmak üzere veri seti giriş değişkenleri
# ve tahmin edilmek istenen çıktı değişkeni birbirinden ayrılmaktadır.

# Özellik değişkenleri (features)
# risk_level hedef değişken olduğu için model girdilerinden çıkarılmıştır.
# X içerisinde modelin tahmin yaparken kullanacağı tüm bağımsız değişkenler bulunur.
X = df_model.drop(
    "risk_level",
    axis=1
)


# Hedef değişken (target)
# Modelin tahmin etmeye çalışacağı risk sınıflarını içerir.
# risk_level değerleri:
# 0 → Düşük Risk
# 1 → Orta Risk
# 2 → Yüksek Risk
# (Label Encoding aşamasında sayısal değerlere dönüştürülecektir.)
y = df_model["risk_level"]

In [ ]:
X.shape, y.shape

In [ ]:
# Hedef değişkeni özelliklerden ayırma
# Modelleme ve outlier analizi sırasında hedef değişken olan risk_level kullanılmayacaktır.
# Bu nedenle bağımsız değişkenlerden (features) çıkarılarak yalnızca model girdileri hazırlanmıştır.

X_outlier = df_model.drop(
    "risk_level",
    axis=1
)


# Sayısal değişkenleri belirleme
# Outlier analizi yalnızca sayısal değişkenler üzerinde uygulanabildiği için,
# veri setindeki int ve float tipindeki kolonlar seçilmiştir.
# Kategorik değişkenler (object/string) bu analize dahil edilmemiştir.

numeric_cols = X_outlier.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

In [ ]:
X = df_model.drop("risk_level", axis=1)

y = df_model["risk_level"]

X.dtypes

In [ ]:
# Veri setindeki özellikleri veri tiplerine göre ayırma
# Makine öğrenmesi öncesinde sayısal ve kategorik değişkenler farklı ön işleme
# adımlarından geçirileceği için kolonlar iki gruba ayrılmaktadır.


# Sayısal özelliklerin belirlenmesi
# int ve float veri tipine sahip kolonlar seçilir.
# Bu kolonlar median ile eksik değer doldurma ve ölçeklendirme işlemlerine tabi tutulacaktır.

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()


# Kategorik özelliklerin belirlenmesi
# Metinsel (object) veri tipine sahip kolonlar seçilir.
# Bu kolonlar most_frequent doldurma ve OneHotEncoder işlemleri ile
# sayısal forma dönüştürülecektir.

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [ ]:
print("Numerical:", numeric_features)
print("Categorical:", categorical_features)

In [ ]:
X.columns

In [ ]:
y.name

In [ ]:
df_model.columns

In [ ]:
y.value_counts()

In [ ]:
X.isnull().sum().sum()

In [ ]:
# Kategorik (metinsel) değişkenleri belirleme
# Veri setindeki object veri tipine sahip kolonları seçer.
# Bu kolonlar sayısal olmadığı için modelleme öncesinde OneHotEncoder gibi
# kodlama yöntemleriyle sayısal forma dönüştürülmeleri gerekir.

X.select_dtypes(
    include="object"
).columns

In [ ]:
# Kategorik değişkenlerdeki benzersiz değer sayılarını kontrol etme
# OneHotEncoder uygulanmadan önce kategorik kolonların kaç farklı kategori içerdiği incelenmektedir.
# Çok fazla benzersiz değere sahip kolonlar model boyutunu artırabileceği için kontrol edilir.

for col in X.select_dtypes(include="object").columns:
    print(f"{col}: {X[col].nunique()} farklı değer")

In [ ]:
print(len(numeric_features))
print(len(categorical_features))

In [ ]:
print("df_model:", df_model.shape)
print("X:", X.shape)

In [ ]:
X.dtypes.value_counts()

In [ ]:
print("Numeric:", len(numeric_features))
print(numeric_features)

print("\nCategorical:", len(categorical_features))
print(categorical_features)

In [ ]:
# Hedef değişken olan risk_level oluşturulurken kullanılan risk göstergelerini
# model girişinden çıkarma.
# Böylece modelin gerçek ihale özelliklerinden risk tahmini yapması sağlanır.

risk_cols = [
    "risk_indicator_count",
    "corr_nocft",
    "corr_proc",
    "corr_decp",
    "corr_singleb",
    "corr_subm",
    "corr_buyer_concentration"
]

# Hedef değişken olan risk_level oluşturulurken kullanılan risk göstergelerini
# model girişinden çıkarma.
# Böylece modelin gerçek ihale özelliklerinden risk tahmini yapması sağlanır.

X = X.drop(
    columns=risk_cols,
    errors="ignore"
)

In [ ]:
# Güncel X üzerinden sayısal ve kategorik kolonları belirleme

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


print("Numeric:", len(numeric_features))
print(numeric_features)

print("\nCategorical:", len(categorical_features))
print(categorical_features)

In [ ]:
# Veri ön işleme (preprocessing) pipeline'ının oluşturulması
# Sayısal ve kategorik değişkenler farklı işlemlerden geçirileceği için
# kolonlar veri tiplerine göre ayrılmaktadır.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


# Sayısal özelliklerin belirlenmesi
# int ve float veri tipindeki kolonlar seçilir.
# Bu değişkenlerde eksik değerler medyan ile doldurulacaktır.

numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()


# Kategorik özelliklerin belirlenmesi
# Object tipindeki kolonlar seçilir.
# Bu değişkenler OneHotEncoder ile sayısal forma dönüştürülecektir.

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()


print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))


# Sayısal değişkenler için pipeline
# Eksik sayısal değerler, uç değerlerden daha az etkilenen medyan değeri ile doldurulur.

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)


# Kategorik değişkenler için pipeline
# Eksik kategoriler en sık görülen değer ile doldurulur.
# OneHotEncoder ile kategorik değişkenler modele uygun sayısal forma çevrilir.
# handle_unknown="ignore" sayesinde eğitim sırasında görülmeyen yeni kategoriler
# test verisinde hata oluşturmaz.

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)


# Sayısal ve kategorik pipeline'larını birleştirme
# Her değişkene uygun ön işleme adımının uygulanmasını sağlar.

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# Özelliklerin (X) ön işleme pipeline'ından geçirilmesi
# Sayısal değişkenlerde eksik değer doldurma ve ölçeklendirme,
# kategorik değişkenlerde ise eksik değer doldurma ve OneHot Encoding
# işlemleri uygulanarak modelin kullanabileceği sayısal formata dönüştürülür.

X_encoded = preprocessor.fit_transform(X)


# Dönüşüm öncesi ve sonrası veri boyutlarını karşılaştırma
# OneHot Encoding işlemi kategorik değişkenleri birden fazla sütuna ayırdığı için
# özellik sayısı dönüşüm sonrasında artabilir.

print("Eski X boyutu:", X.shape)
print("Encoded X boyutu:", X_encoded.shape)


In [ ]:
# Hedef değişkenin (risk_level) sayısal forma dönüştürülmesi
# Makine öğrenmesi algoritmaları kategorik metin değerleri doğrudan işleyemediği için
# risk sınıfları sayısal etiketlere dönüştürülmektedir.

from sklearn.preprocessing import LabelEncoder

# LabelEncoder nesnesi oluşturma
label_encoder = LabelEncoder()


# Risk seviyelerini sayısal sınıflara dönüştürme
# Örneğin:
# Düşük Risk  -> 0
# Orta Risk   -> 1
# Yüksek Risk -> 2

y_encoded = label_encoder.fit_transform(y)


# Oluşturulan sınıf eşleşmelerini görüntüleme
# Model çıktılarının hangi risk seviyesine karşılık geldiğini kontrol etmek için kullanılır.

print("Sınıf eşleşmeleri:")

for class_name, class_number in zip(
    label_encoder.classes_,
    range(len(label_encoder.classes_))
):
    print(class_name, "->", class_number)


print("\nYeni y boyutu:", y_encoded.shape)

In [ ]:
# Eğitim ve test veri setlerinin ayrılması
# Veri seti modelin öğrenmesi ve performansının değerlendirilmesi için
# eğitim ve test olmak üzere iki parçaya ayrılmaktadır.

from sklearn.model_selection import train_test_split


# Veri setinin %80'i eğitim, %20'si test için ayrılır.
# random_state=42 kullanılarak aynı sonuçların tekrar üretilebilmesi sağlanır.
# stratify=y ile risk sınıflarının (Düşük, Orta, Yüksek Risk) eğitim ve test
# kümelerinde aynı oranlarda dağılması sağlanır.

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 🧠 10. Machine Learning Models / Makine Öğrenmesi Modelleri

This section trains and compares different machine learning models.

Bu bölümde farklı makine öğrenmesi modelleri eğitilmekte ve karşılaştırılmaktadır.